# 04 Results

Result tables, figures, hypothesis tests and failure analysis for the report.

1. What we compare (runs and metrics)
2. Main results on test users
3. Validation vs. test
4. Per-user scores
5. Hypotheses H1–H3
6. Why the numbers are low
7. Failure analysis
8. Summary

Final numbers are for **test users** only (90 users). Validation users were used to pick settings in `03_models.ipynb`.

**Every score in this notebook is measured against the answer key:** the hidden ratings of each test user (the beers they actually rated after their first 5). The baseline is not the answer key; it is the simple model the other runs are compared with.

In [ ]:
import os
from pathlib import Path

# Run from the project folder so `src` can be imported
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.data import load_beers
from src.metrics import user_scores
from src.models import popular_beers, recommend_bpr, recommend_popular, train_bpr
from src.split import load_split, non_nordic_sample, nordic_train

# Settings picked on validation users in 03_models.ipynb
BEST_FACTORS = 128
BEST_BONUS = 1.0

METRICS = ["recall@10", "ndcg@10", "coverage@10"]
RUNS = ["baseline", "B", "C", "D", "E"]
RUN_LABELS = {
    "baseline": "Baseline: most popular",
    "B": "B: BPR, all countries",
    "C": "C: BPR + home bonus",
    "D": "D: BPR, Nordic only",
    "E": "E: BPR, non-Nordic sample",
}
ALPHA = 0.05

# Colours: one hue for single-series charts, fixed order for model comparisons
BLUE, ORANGE, AQUA, YELLOW, VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#4a3aa7"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e5e5e5", "axes.axisbelow": True})

results = pd.read_csv("results/model_results.csv")

## 1. What we compare

| Run | Model | Training data | Used for |
|---|---|---|---|
| Baseline | Most popular beers (number of ratings) | All countries except Norway | Comparison point |
| B | BPR, 128 factors | All countries except Norway | H1 |
| C | B + home-country bonus (w = 1.0) | Same as B | H2 |
| D | BPR, 128 factors | Sweden, Denmark, Finland (774,075 ratings) | H3 |
| E | BPR, 128 factors | Random non-Nordic sample, same number of ratings as D | H3 |

All BPR runs also see the first 5 ratings of each Norwegian test user. No run recommends a beer the user already rated.

| Metric | What it measures | How to read it |
|---|---|---|
| Recall@10 | Share of the user's hidden beers that are in the top 10 | Higher is better. Max is 10 / number of hidden beers |
| NDCG@10 | Like recall, but hits near the top count more | Higher is better, 0–1. **Main metric** |
| Coverage@10 | Share of all beers shown to at least one user | Higher means more variety. Not a quality measure on its own |

## 2. Main results (test users)

In [ ]:
test_results = results[results["group"] == "test"].set_index("run").loc[RUNS, METRICS]
test_results.to_csv("results/test_results.csv")

test_table = test_results.rename(index=RUN_LABELS)
test_table.style.format({"recall@10": "{:.4f}", "ndcg@10": "{:.3f}", "coverage@10": "{:.5f}"}) \
    .highlight_max(color="#d6e6f9")

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(13, 4))
for ax, metric in zip(axes, METRICS):
    values = test_results[metric]
    bars = ax.bar(values.index, values, color=BLUE, width=0.6)
    ax.bar_label(bars, fmt="%.5f" if metric == "coverage@10" else "%.3f" if metric == "ndcg@10" else "%.4f",
                 fontsize=8, color="#52514e")
    ax.set_title(metric)
    ax.grid(axis="x", visible=False)
fig.suptitle("Test users (n = 90)")
fig.tight_layout()
fig.savefig("results/metrics_by_run.png", dpi=150)

## 3. Validation vs. test

The same settings scored on both user groups. Large differences show how much the results move with only 90 users per group.

In [ ]:
ndcg_by_group = results.pivot(index="run", columns="group", values="ndcg@10").loc[RUNS, ["validation", "test"]]
ndcg_by_group["difference"] = ndcg_by_group["test"] - ndcg_by_group["validation"]
ndcg_by_group.rename(index=RUN_LABELS).round(3)

## 4. Per-user scores

The hypothesis tests and the failure analysis need a score for each user, not only the mean. We retrain the models exactly as in `03_models.ipynb` (same data, settings and seed) and score each test user. Training takes a few minutes.

The check below compares the means with `model_results.csv`. Small differences can come from BPR training on several threads.

In [ ]:
train, history, hidden = load_split()
test_hidden = hidden[hidden["group"] == "test"]

bpr_all = train_bpr(train, history, factors=BEST_FACTORS)
nordic = nordic_train(train)
bpr_nordic = train_bpr(nordic, history, factors=BEST_FACTORS)
bpr_sample = train_bpr(non_nordic_sample(train, len(nordic)), history, factors=BEST_FACTORS)

recommendations = {
    "baseline": recommend_popular(popular_beers(train), history),
    "B": recommend_bpr(bpr_all, history),
    "C": recommend_bpr(bpr_all, history, home_bonus=BEST_BONUS),
    "D": recommend_bpr(bpr_nordic, history),
    "E": recommend_bpr(bpr_sample, history),
}
scores = {run: user_scores(recs, test_hidden).set_index("user_id") for run, recs in recommendations.items()}
ndcg = pd.DataFrame({run: run_scores["ndcg"] for run, run_scores in scores.items()})
recall = pd.DataFrame({run: run_scores["recall"] for run, run_scores in scores.items()})

pd.DataFrame({"ndcg@10 (03)": test_results["ndcg@10"], "ndcg@10 (here)": ndcg.mean()}).round(3)

## 5. Hypotheses

Each hypothesis compares two runs on the **same 90 test users**. We use a paired Wilcoxon signed-rank test on NDCG@10 per user: it asks whether one run beats the other for most users, not only on average. The difference counts as significant if p < 0.05.

- **Supported:** the new run is better and p < 0.05
- **Not significant:** the difference may be chance
- **Rejected:** the new run is worse and p < 0.05

In [ ]:
HYPOTHESES = [
    ("H1", "Does BPR on foreign data beat the popular list?", "baseline", "B"),
    ("H2", "Does a home-country bonus help?", "B", "C"),
    ("H3", "Is Nordic data better than the same amount of non-Nordic data?", "E", "D"),
]


def verdict(change, p_value):
    """Plain-language outcome of a hypothesis test."""
    if p_value >= ALPHA:
        return "Not significant"
    return "Supported" if change > 0 else "Rejected"


rows = []
for hypothesis, question, reference, run in HYPOTHESES:
    difference = ndcg[run] - ndcg[reference]
    p_value = wilcoxon(ndcg[run], ndcg[reference]).pvalue
    rows.append({
        "hypothesis": hypothesis,
        "question": question,
        "comparison": f"{run} vs {reference}",
        "ndcg reference": ndcg[reference].mean(),
        "ndcg new": ndcg[run].mean(),
        "ndcg change": ndcg[run].mean() / ndcg[reference].mean() - 1,
        "recall change": recall[run].mean() / recall[reference].mean() - 1,
        "users better": (difference > 0).sum(),
        "users worse": (difference < 0).sum(),
        "p-value": p_value,
        "result": verdict(difference.mean(), p_value),
    })
hypothesis_table = pd.DataFrame(rows).set_index("hypothesis")
hypothesis_table.to_csv("results/hypothesis_tests.csv")
hypothesis_table.style.format({
    "ndcg reference": "{:.3f}", "ndcg new": "{:.3f}", "ndcg change": "{:+.1%}",
    "recall change": "{:+.1%}", "p-value": "{:.3f}",
})

## 6. Why the numbers are low

Recall@10 looks very low, but most test users have far more than 10 hidden beers, so even a perfect top 10 could only find a small share of them. Some hidden beers are also not in the training data at all, so no model can recommend them.

In [ ]:
train_beers = set(train["beer_id"])
hidden_by_user = test_hidden.groupby("user_id")
n_hidden = hidden_by_user.size()
max_recall = (10 / n_hidden).clip(upper=1)
share_unreachable = hidden_by_user["beer_id"].apply(lambda beers: (~beers.isin(train_beers)).mean())

pd.DataFrame({
    "median": [n_hidden.median(), max_recall.median(), share_unreachable.median()],
    "mean": [n_hidden.mean(), max_recall.mean(), share_unreachable.mean()],
}, index=["hidden beers per user", "best possible recall@10", "share of hidden beers not in train"]).round(3)

## 7. Failure analysis

Which test users get poor recommendations, and why? We describe each test user and look at their NDCG@10 for every run. Each score is measured against the user's hidden ratings (the answer key); the runs are shown side by side only to see if they fail for the same users.

The tables show all five runs. The first figure shows the three runs trained on all data (baseline, B, C), and a second figure compares D and E (H3).

Hidden ratings are used here only to describe the users, never to train anything.

In [ ]:
styles = load_beers().set_index("beer_id")["style"]
test_ratings = pd.concat([history[history["group"] == "test"], test_hidden])

users = pd.DataFrame({
    "n_ratings": test_ratings.groupby("user_id").size(),
    "share_norwegian": hidden_by_user["beer_country"].apply(lambda country: (country == "Norway").mean()),
    "share_unreachable": share_unreachable,
    "top_style": test_ratings["beer_id"].map(styles).groupby(test_ratings["user_id"]).agg(lambda style: style.mode().iloc[0]),
})
users = users.join(ndcg.add_suffix("_ndcg"))
users.describe().round(3)

### Users with no hits

How do users where the best run (C) finds nothing in the top 10 differ from the rest?

In [ ]:
no_hits = (users["C_ndcg"] == 0).map({True: "no hits", False: "at least one hit"}).rename("C result")
print(f"{(users['C_ndcg'] == 0).mean():.0%} of test users get no hits from run C")
no_hit_table = users.groupby(no_hits)[["n_ratings", "share_norwegian", "share_unreachable"]].median()
no_hit_table.insert(0, "users", no_hits.value_counts())
no_hit_table.round(3)

### NDCG@10 by user activity and taste for Norwegian beer

Users are split into four equal-sized groups (quartiles): by number of ratings, and by share of Norwegian beers among their hidden ratings.

In [ ]:
QUARTILES = ["low", "mid-low", "mid-high", "high"]
# rank first so ties do not break the four equal groups
users["activity"] = pd.qcut(users["n_ratings"].rank(method="first"), 4, labels=QUARTILES)
users["norwegian_taste"] = pd.qcut(users["share_norwegian"].rank(method="first"), 4, labels=QUARTILES)

GROUPINGS = {"activity": "Number of ratings", "norwegian_taste": "Share of Norwegian beers"}
NDCG_COLUMNS = [f"{run}_ndcg" for run in RUNS]
group_tables = {
    column: users.groupby(column, observed=True)[NDCG_COLUMNS].mean().set_axis(RUN_LABELS.values(), axis=1)
    for column in GROUPINGS
}
pd.concat(group_tables, names=["grouping", "quartile"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (column, title) in zip(axes, GROUPINGS.items()):
    main_runs = [RUN_LABELS[run] for run in ["baseline", "B", "C"]]
    group_tables[column][main_runs].plot.bar(ax=ax, rot=0, color=[BLUE, ORANGE, AQUA], width=0.75,
                                             edgecolor="white", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("quartile")
    ax.set_ylabel("mean NDCG@10 (vs. hidden ratings)")
    ax.grid(axis="x", visible=False)
axes[1].get_legend().remove()
axes[0].legend(title="run", frameon=False)
fig.suptitle("Test users by group (n = 90, about 22 per quartile)")
fig.tight_layout()
fig.savefig("results/failure_analysis.png", dpi=150)

### D vs. E: who gains from Nordic data?

Same quartiles, but for the two runs trained on the same amount of data (H3). If Nordic data works because Nordic taste is close to Norwegian taste, D should help most for users who drink a lot of Norwegian beer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (column, title) in zip(axes, GROUPINGS.items()):
    group_tables[column][[RUN_LABELS["D"], RUN_LABELS["E"]]].plot.bar(
        ax=ax, rot=0, color=[VIOLET, YELLOW], width=0.6, edgecolor="white", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("quartile")
    ax.set_ylabel("mean NDCG@10 (vs. hidden ratings)")
    ax.grid(axis="x", visible=False)
axes[1].get_legend().remove()
axes[0].legend(title="run", frameon=False)
fig.suptitle("D vs. E on test users (n = 90, about 22 per quartile)")
fig.tight_layout()
fig.savefig("results/failure_analysis_d_vs_e.png", dpi=150)

### NDCG@10 by favourite style

Only styles that are the favourite of at least 3 test users.

In [ ]:
style_table = users.groupby("top_style").agg(users=("C_ndcg", "size"), **{
    f"{run}_ndcg": (f"{run}_ndcg", "mean") for run in RUNS
})
style_table[style_table["users"] >= 3].sort_values("C_ndcg").round(3)

## 8. Summary

| | Result | What it means |
|---|---|---|
| **H1** | Rejected on test users | BPR on foreign data (B) does worse than the popular list (NDCG 0.25 vs. 0.33, worse for 43 of 90 users, better for 23). On validation users B was slightly better, so the result is not stable, but there is no support for H1. |
| **H2** | Supported | The home-country bonus (C) is the best run overall: NDCG +43 % over B, better for 57 users and worse for 13. It also beats the baseline. |
| **H3** | Supported | With the same number of ratings, Nordic data (D) gives much better recommendations than non-Nordic data (E): NDCG +62 %. |

**Answers to the research questions**
- **RQ1:** Plain BPR on data from other countries does not beat recommending the most popular beers.
- **RQ2:** Yes. Knowing where the beer comes from (H2) and using culturally similar countries (H3) both improve the recommendations clearly.

**Failure analysis**
- Users with few ratings get the worst results. Users with no hits from C have a median of 26 ratings, compared with 342 for the rest. Part of this comes from the metric: a user with few hidden beers has fewer chances to get a hit.
- The home bonus helps most for users who drink a lot of Norwegian beer (the highest quartile goes from 0.14 to 0.27). For users who drink little Norwegian beer it does not help.
- Fans of IPA and pale lager get the best recommendations. Fans of Belgian ales, brown ales and imperial stouts get the worst, but these groups are small (3–10 users).
- Recall@10 is low for all runs because the median user has 115 hidden beers, so even a perfect top 10 reaches at most about 9 %.

**Things to keep in mind in the report**
- Only 90 test users, so single numbers move a lot between validation and test (section 3).
- w = 1.0 was the largest bonus we tried, so an even larger bonus might work better.
- D has 1,951 users with many ratings each, while E has 18,062 users with few each. Part of the D–E difference may come from this, not only from culture.
- The bonus makes the lists less varied: C has lower coverage than B.